# 72 — Train SASRec recall channel + ablate vs union (P0)

Trains the dialog-conditioned, content-fused SASRec on Colab GPU and measures
recall@100 lift as a 4th channel in `wrrf_union_v1` (`use_sasrec`).

Run order:
1. Cell 1 — setup (clones the branch, installs deps, mounts Drive, symlinks the cache).
2. Cell 2 — train SASRec (writes `sasrec_v1/sasrec.pt` to the Drive cache).
3. Cell 3 — recall ablation (union vs union+SASRec on the full dev split).

References: spec `docs/superpowers/specs/2026-05-27-sasrec-recall-channel-design.md`,
plan `docs/superpowers/plans/2026-05-27-sasrec-recall-channel.md`,
memory `project_sasrec_p0_implemented_2026_05_27.md`.


In [ ]:
# 1) Setup. Disable JAX GPU preallocation BEFORE any import pulls JAX in.
# datasets/transformers import JAX transitively; JAX grabs ~75% of VRAM on
# first use, so the KERNEL ends up hogging the GPU and the cell-3 !python
# subprocess OOMs. This is why nb 72 OOM'd while nb 70/71 (which set these)
# did not. If the kernel already imported JAX, RESTART RUNTIME for this to take.
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
os.environ.setdefault('TF_FORCE_GPU_ALLOW_GROWTH', 'true')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=False)

BRANCH = 'recall-union-lgbm'  # G2: 3-channel union pool + new session features + album_name fix
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
src = f'{DRIVE_BASE}/recsys2026_retrieval_v2_cache'
dst = f'{LOCAL_BASE}/retrieval_v2'
if os.path.islink(dst): os.unlink(dst)
elif os.path.exists(dst):
    import shutil; shutil.rmtree(dst)
os.symlink(src, dst)

# Retrieval stack (bm25->bm25s, dense->sentence-transformers/peft) is imported
# eagerly by mcrs.retrieval_modules, so its deps are required even for the
# LGBM build. Matches nb 71's proven set + lightgbm/scikit-learn.
!pip install -q --upgrade 'transformers>=4.40' 'accelerate>=0.30' 'peft>=0.11' \
    'datasets' 'pandas<3.0' 'tqdm' 'huggingface_hub' 'sentence-transformers>=3.0' \
    'FlagEmbedding>=1.3' 'bm25s' 'lightgbm' 'scikit-learn'

In [ ]:
# 2) Train SASRec on the train split (Colab GPU). Writes
# experiments/cache/retrieval_v2/sasrec/sasrec_v1/sasrec.pt -> Drive (via the
# cell-1 symlink), so it persists across runtimes.
#
# Per-epoch output: train_loss, val_loss, val_recall@{20,100} (val held out
# session-disjoint from train via SHA1(session_id)). End-of-run: standalone
# TEST recall@{20,100} (sanity number; model selection uses val, not test).
#
# Pre-flight — run ONCE to confirm the CLAP audio column name on the dataset.
# If it isn't 'audio-laion_clap', edit CLAP_COL in scripts/train_sasrec.py
# and re-commit + git pull on Colab.
#   from datasets import load_dataset
#   ds = load_dataset('talkpl-ai/TalkPlayData-Challenge-Track-Embeddings', split='all_tracks')
#   print([c for c in ds.column_names if 'clap' in c.lower() or 'audio' in c.lower()])
!cd /content/recsys2026 && python -u scripts/train_sasrec.py \
    --cache-dir /content/recsys2026/experiments/cache \
    --out sasrec_v1 \
    --epochs 5


In [ ]:
# 3) SASRec recall ablation: union without vs with the SASRec channel
# (use_sasrec). Builds dev data with the user-turns dialog, prints the
# dialog-length distribution vs bge-base-en's 512 cap, then reports
# recall@{20,100} for both unions plus the delta. Decision gate:
# union+SASRec recall@100 >= baseline + ~0.03 -> proceed to P1 (feed the
# SASRec score into the LGBM as the discriminating relevance feature).
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer
from mcrs.db_item.music_catalog import MusicCatalogDB
from mcrs.retrieval_modules import load_retrieval_module
from mcrs.retrieval_modules.sasrec_model import build_user_dialog

# Shared constants (self-contained — no dependency on earlier cells).
ITEM_DB = 'talkpl-ai/TalkPlayData-Challenge-Track-Metadata'
CORPUS = ['track_name', 'artist_name', 'album_name']
CACHE_DIR = '/content/recsys2026/experiments/cache'

item_db = MusicCatalogDB(ITEM_DB, ['all_tracks'], CORPUS)
dev = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')
queries, golds, user_ids, played, user_dialogs = [], [], [], [], []
N_SASREC_EVAL = None  # None = full dev; set an int to cap for a smoke run
for sess in dev:
    if N_SASREC_EVAL is not None and len(queries) >= N_SASREC_EVAL:
        break
    df = pd.DataFrame(sess['conversations'])
    for _, music in df[df['role'] == 'music'].iterrows():
        if N_SASREC_EVAL is not None and len(queries) >= N_SASREC_EVAL:
            break
        tn = int(music['turn_number'])
        prior = df[(df['turn_number'] < tn) |
                   ((df['turn_number'] == tn) & (df['role'] == 'user'))]
        lines = []
        for _, t in prior.iterrows():
            role = 'assistant' if t['role'] == 'music' else t['role']
            content = item_db.id_to_metadata(t['content']) if t['role'] == 'music' else t['content']
            lines.append(f'{role}: {content}')
        queries.append(chr(10).join(lines))
        user_dialogs.append(build_user_dialog(prior.to_dict('records')))
        golds.append(music['content'])
        user_ids.append(sess.get('user_id'))
        played.append(list(df[(df['role'] == 'music') & (df['turn_number'] < tn)]['content']))
print('[sasrec eval] built', len(queries), 'dev turns')

# Dialog-length check: how many user-dialogs exceed bge-base-en's 512-token cap?
tok = AutoTokenizer.from_pretrained('BAAI/bge-base-en-v1.5')
lens = [len(tok.encode(d)) for d in user_dialogs]
print('[sasrec eval] user-dialog tokens: median', int(np.median(lens)),
      ' p95', int(np.percentile(lens, 95)),
      ' frac>512:', round(float(np.mean([l > 512 for l in lens])), 4))

def recall_at(cands, k):
    return float(np.mean([1.0 if g in c[:k] else 0.0 for c, g in zip(cands, golds)]))

base = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS,
                             CACHE_DIR, extra_config={})
sas = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS,
                            CACHE_DIR, extra_config={'use_sasrec': True, 'w_sasrec': 1.0})
ctx = [{'history_tids': p, 'user_dialog': ud} for p, ud in zip(played, user_dialogs)]
cb = base.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)
cs = sas.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)
print('=== SASRec channel recall ablation (n=' + str(len(golds)) + ', FULL dev) ===')
print('  union (3-chan) : recall@20=' + str(round(recall_at(cb, 20), 4)) +
      ' @100=' + str(round(recall_at(cb, 100), 4)))
print('  union + SASRec : recall@20=' + str(round(recall_at(cs, 20), 4)) +
      ' @100=' + str(round(recall_at(cs, 100), 4)) + '   (G1 gate 0.46)')
print('  delta recall@100 :', round(recall_at(cs, 100) - recall_at(cb, 100), 4))
